In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from datasets import load_dataset

In [2]:
VOCAB_SIZE = 12000
EMBEDDING_DIM = 100
HIDDEN_DIM = 128
OUTPUT_DIM = 13
BATCH_SIZE = 64
EPOCHS = 5
MAX_LEN = 60

In [3]:
print("Loading dair-ai/emotion...")
dataset = load_dataset("dair-ai/emotion")

Loading dair-ai/emotion...


README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

C:\Users\HP USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP USER\.cache\huggingface\hub\datasets--dair-ai--emotion. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


split/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.03MB            

split/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

split/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  129kB            

split/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [4]:
label_names = dataset["train"].features["label"].names
X_train_raw = list(dataset["train"]["text"])
y_train_raw = list(dataset["train"]["label"])

In [5]:
if "validation" in dataset:
    X_val_raw = list(dataset["validation"]["text"])
    y_val_raw = list(dataset["validation"]["label"])
else:
    total_len = len(X_train_raw)
    val_size = int(total_len * 0.1)

    X_val_raw = X_train_raw[-val_size:]
    y_val_raw = y_train_raw[-val_size:]

    X_train_raw = X_train_raw[:-val_size]
    y_train_raw = y_train_raw[:-val_size]

y_train = np.array(y_train_raw, dtype=np.int32)
y_val = np.array(y_val_raw, dtype=np.int32)

print(f"Training samples: {len(X_train_raw)}")
print(f"Validation samples: {len(X_val_raw)}")

print("\nTokenizing text...")

Training samples: 16000
Validation samples: 2000

Tokenizing text...


In [6]:
tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<OOV>"
)
tokenizer.fit_on_texts(X_train_raw)
X_train_seq = tokenizer.texts_to_sequences(X_train_raw)
X_val_seq = tokenizer.texts_to_sequences(X_val_raw)
print("Tokenization completed.")
print("First training sequence:", X_train_seq[0])
print("First validation sequence:", X_val_seq[0])
print("\nPadding sequences...")

Tokenization completed.
First training sequence: [2, 139, 3, 679]
First validation sequence: [17, 8, 157, 260, 4, 343, 16, 51, 19, 212, 11289, 50, 10, 13, 533]

Padding sequences...


In [8]:
X_train = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)
X_val = pad_sequences(
    X_val_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)
print("Preprocessing complete.")
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("\nBuilding LSTM model...")

Preprocessing complete.
X_train shape: (16000, 60)
X_val shape: (2000, 60)

Building LSTM model...


In [9]:
model = Sequential([
    tf.keras.layers.Input(shape=(MAX_LEN,)),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    LSTM(
        units=HIDDEN_DIM,
        return_sequences=False
    ),
    Dropout(0.3),
    Dense(
        units=OUTPUT_DIM,
        activation="softmax"
    )
])

In [10]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Layer (type)             ┃ Output Shape      ┃   Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ embedding (Embedding)    │ (None, 60, 100)   │ 1,200,000 │
├──────────────────────────┼───────────────────┼───────────┤
│ lstm (LSTM)              │ (None, 128)       │   117,248 │
├──────────────────────────┼───────────────────┼───────────┤
│ dropout (Dropout)        │ (None, 128)       │         0 │
├──────────────────────────┼───────────────────┼───────────┤
│ dense (Dense)            │ (None, 13)        │     1,677 │
└──────────────────────────┴───────────────────┴───────────┘

 Total params: 1,318,925 (5.03 MB)

 Trainable params: 1,318,925 (5.03 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)

Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 21s 75ms/step - accuracy: 0.3152 - loss: 1.6564 - val_accuracy: 0.3530 - val_loss: 1.5927
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 18s 73ms/step - accuracy: 0.3252 - loss: 1.5949 - val_accuracy: 0.2750 - val_loss: 1.5945
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 67ms/step - accuracy: 0.3277 - loss: 1.5889 - val_accuracy: 0.3520 - val_loss: 1.5855
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 66ms/step - accuracy: 0.3301 - loss: 1.5846 - val_accuracy: 0.3520 - val_loss: 1.5801
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 66ms/step - accuracy: 0.3267 - loss: 1.5812 - val_accuracy: 0.3520 - val_loss: 1.5827


In [12]:
label_mapping = {
    0: "sadness",
    1: "happiness",
    2: "love",
    3: "anger",
    4: "fear",
    5: "surprise",
    6: "shame",
    7: "guilt",
    8: "disgust",
    9: "confusion",
    10: "boredom",
    11: "relief",
    12: "sarcasm"
}

In [13]:
def predict_emotion(text):
    sequence = tokenizer.texts_to_sequences([text])
    padded_sequence = pad_sequences(
        sequence,
        maxlen=MAX_LEN,
        padding="post",
        truncating="post"
    )
    prediction = model.predict(
        padded_sequence,
        verbose=0
    )
    predicted_class = np.argmax(prediction[0])
    emotion = label_mapping[predicted_class]
    confidence = prediction[0][predicted_class]
    return emotion, confidence

In [14]:
def get_sentiment(emotion):
    if emotion in [
        "happiness",
        "love",
        "relief"
    ]:
        return "Positive"
    elif emotion in [
        "sadness",
        "anger",
        "fear",
        "shame",
        "guilt",
        "disgust",
        "boredom"
    ]:
        return "Negative"
    else:
        return "Neutral/Complex"

In [15]:
def predict_emotion_and_sentiment(text):
    emotion, confidence = predict_emotion(text)
    sentiment = get_sentiment(emotion)
    return emotion, sentiment, confidence
text = "I am extremely angry right now."
emotion, sentiment, confidence = predict_emotion_and_sentiment(text)

In [17]:
print("\n******************************")
print("       MODEL PREDICTION")
print("******************************")
print("Input Text:", text)
print("Predicted Emotion:", emotion)
print("Sentiment:", sentiment)
print("Confidence:", round(float(confidence) * 100, 2), "%")


******************************
       MODEL PREDICTION
******************************
Input Text: I am extremely angry right now.
Predicted Emotion: happiness
Sentiment: Positive
Confidence: 32.1 %
